# 03 — Vector Store
Verifikasi vectors yang sudah disimpan ke Chroma local.
Vectors tidak perlu di-generate ulang — langsung load dari disk.

## 0. Load Chroma dari Disk
Load vector store yang sudah tersimpan di folder `vector_db/`.
Tidak perlu embed ulang!

In [1]:
import chromadb

client = chromadb.PersistentClient(path="../vector_db")
collection = client.get_collection(name="phis_sds")

print(f"Collection     : {collection.name}")
print(f"Total vectors  : {collection.count()}")

Collection     : phis_sds
Total vectors  : 666


## 1. Peek Data — Lihat Isi Vector Store
Lihat sample data yang tersimpan di Chroma.

In [2]:
# Lihat 3 data pertama
result = collection.get(
    ids=["0", "1", "2"],
    include=["documents", "metadatas"]
)

for i in range(3):
    print(f"{'='*55}")
    print(f"ID       : {result['ids'][i]}")
    print(f"Halaman  : {result['metadatas'][i]['page']}")
    print(f"Konten   : {result['documents'][i][:150]}")
print(f"{'='*55}")

ID       : 0
Halaman  : 0
Konten   : SYSTEM DESIGN
SPECIFICATION (SDS)
Pharmacy Information System (PhIS)
Special Approval Medicine (SAM)
DOCUMENT DATE : 06/05/2025
DOCUMENT VERSION : 1.0
ID       : 1
Halaman  : 1
Konten   : System Design Specification (SDS)
Ref: PhIS/SDS/SAM Page i
DOCUMENT INFORMATION
This document details the system design for the Pharmacy Information S
ID       : 2
Halaman  : 1
Konten   : design emphasizes user-friendliness, accessibility, and efficiency, ensuring the system
accommodates the varied needs of users while requiring minimal


## 2. Semantic Search — Query Pertama!
Test query natural language ke vector store.
Chroma akan cari chunks yang paling relevan secara makna.

In [5]:
from sentence_transformers import SentenceTransformer

# Load embedding model
model = SentenceTransformer("thenlper/gte-large")

def search(query, n_results=3):
    # Embed query
    query_vector = model.encode(query).tolist()
    
    # Search di Chroma
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )
    
    print(f"Query: '{query}'")
    print(f"{'='*55}")
    for i in range(n_results):
        score = 1 - results['distances'][0][i]
        page = results['metadatas'][0][i]['page']
        doc = results['documents'][0][i][:200]
        print(f"Rank {i+1} | Score: {score:.4f} | Halaman: {page}")
        print(f"{doc}")
        print(f"{'-'*55}")

# Test query pertama!
search("replenishment process pharmacy")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 5303.82it/s]


Query: 'replenishment process pharmacy'
Rank 1 | Score: 0.8660 | Halaman: 125
must revise the returned SAM Request. (Date Selection)
 Once Secretariat users have sent the SAM Request back for
revision, facility level users shall be notified about the returned
request in their 
-------------------------------------------------------
Rank 2 | Score: 0.8535 | Halaman: 82
functions as following:
a. On click of ‘Endorse’
 The record status is updated to ‘Pending Review by
Pharmacist’.
 Record will flow to the Pharmacist stage for review.
 Task list/ Pending notificat
-------------------------------------------------------
Rank 3 | Score: 0.8513 | Halaman: 83
requester.
c. On click of ‘Review’
 System shall capture the details of the pharmacist that
reviews the SAM Request (Remarks, Name, Designation,
Discipline, Reviewed Date)
o For MOH Hospital faciliti
-------------------------------------------------------


In [6]:
search("replenishment process pharmacy")

Query: 'replenishment process pharmacy'
Rank 1 | Score: 0.8660 | Halaman: 125
must revise the returned SAM Request. (Date Selection)
 Once Secretariat users have sent the SAM Request back for
revision, facility level users shall be notified about the returned
request in their 
-------------------------------------------------------
Rank 2 | Score: 0.8535 | Halaman: 82
functions as following:
a. On click of ‘Endorse’
 The record status is updated to ‘Pending Review by
Pharmacist’.
 Record will flow to the Pharmacist stage for review.
 Task list/ Pending notificat
-------------------------------------------------------
Rank 3 | Score: 0.8513 | Halaman: 83
requester.
c. On click of ‘Review’
 System shall capture the details of the pharmacist that
reviews the SAM Request (Remarks, Name, Designation,
Discipline, Reviewed Date)
o For MOH Hospital faciliti
-------------------------------------------------------


In [7]:
search("user interface design SAM request form")

Query: 'user interface design SAM request form'
Rank 1 | Score: 0.8955 | Halaman: 41
System Design Specification (SDS)
Ref: PhIS/SDS/SAM Page 30
Request
 Navigate to ‘Overall Request Summary’ tab to view overall list of SAM Request
by whole facility or by same role.
 Navigate to ‘Bu
-------------------------------------------------------
Rank 2 | Score: 0.8949 | Halaman: 49
5. Add New SAM Request Click on ‘Add New’ button to create New SAM Request. This is applicable
for user role with requester rights. Refer to UI-PHIS-SAM-REQ-02 for the
creation details.
Input Validati
-------------------------------------------------------
Rank 3 | Score: 0.8942 | Halaman: 19
System Design Specification (SDS)
Ref: PhIS/SDS/SAM Page 8
3.0 FUNCTIONAL DESIGN
3.1. User Interface Design and Data Mapping
In this section, the user interface (UI) will be designed based on the use 
-------------------------------------------------------


In [8]:
search("database table SAM")

Query: 'database table SAM'
Rank 1 | Score: 0.8796 | Halaman: 206
System Design Specification (SDS)
Ref: PhIS/SDS/SAM Page 195
We are pleased to invite you to join the SAM System. Click on the link below to
register your account.
https://www.SAM.com/registration-lin
-------------------------------------------------------
Rank 2 | Score: 0.8780 | Halaman: 209
SAM Dataset under Appendix.
No Dropdown
23. Professional Registration
No
Professional Registration No of the user.
Mandatory when Professional Type is
selected.
No Text field
24. Account Valid From / 
-------------------------------------------------------
Rank 3 | Score: 0.8778 | Halaman: 104
System Design Specification (SDS)
Ref: PhIS/SDS/SAM Page 93
3.2.1.4. SAM Approval
a. SAM Approval Listing
Table 12 : Functional Design – SAM Approval Listing
UI Reference UI-PHIS-SAM-APR-01
Use Case R
-------------------------------------------------------
